# SD 1.5 (.safetensors) → Qualcomm QNN `qnn2.28_min` — Colab

Bir **safetensors indirme linki** girin → dönüştürün → **Hugging Face reponuza** yükleyin.
Çıktı: `<isim>_qnn2.28_min.zip` (Ruya / Local Dream ile içe aktarılır).

### Önce oku (önemli):
1. **Runtime → Change runtime type → High-RAM** seçin. Ücretsiz katman (12 GB) 512px'te OOM olabilir; **Colab Pro / High-RAM (25–51 GB)** önerilir.
2. **QNN SDK 2.28 otomatik indirilemez** (Qualcomm lisansı). Bir kez indirip Google Drive'a koyun:
   - Qualcomm AI Hub / QPM → `Qualcomm AI Engine Direct SDK 2.28` (v2.28.0.241029)
   - Drive'a `.zip` olarak yükleyin, aşağıda yolunu verin (mount edip açacağız).
3. HF token'ınızı Colab **Secrets** (🔑 sol menü) içine `HF_TOKEN` adıyla ekleyin (write izinli).
4. civitai linki token istiyorsa Secrets'a `CIVITAI_TOKEN` ekleyin.

Hücreleri **sırayla** çalıştırın.

## 1) Ayarlar (buradan doldurun)

In [ ]:
#@title Dönüşüm ayarları { display-mode: "form" }
SAFETENSORS_URL = ""  #@param {type:"string"}
MODEL_NAME = "MyModel"  #@param {type:"string"}
TIER = "min"  #@param ["min", "mid", "high"]
RESOLUTIONS = "512x512"  #@param ["512x512", "512x512,512x768,768x512"] {allow-input: true}

#@markdown **Hugging Face'e yükleme** (isteğe bağlı; boş bırakılırsa sadece indirilir)
HF_REPO = ""  #@param {type:"string"}
HF_PRIVATE = False  #@param {type:"boolean"}

#@markdown **QNN SDK 2.28 .zip'inin Google Drive içindeki yolu**
QNN_SDK_ZIP_ON_DRIVE = "/content/drive/MyDrive/qnn/v2.28.0.241029.zip"  #@param {type:"string"}

assert SAFETENSORS_URL, "SAFETENSORS_URL boş olamaz"
print("Ayarlar tamam:", MODEL_NAME, TIER, RESOLUTIONS)

## 2) Depoyu çek + Python bağımlılıkları + MNN

In [ ]:
%cd /content
![ -d sd-qnn ] || git clone --branch claude/qnn-model-conversion-snapdragon7-rsk8og https://github.com/matrixportalx/Sd-1.5-Converting-to-Qualcomm-QNN-Model.git sd-qnn
%cd /content/sd-qnn
!pip -q install -r requirements.txt
# MNN converter (text_encoder + vae -> .mnn). pip paketi 'mnnconvert' komutunu getirir.
!pip -q install MNN
import os
os.environ["MNNCONVERT"] = "mnnconvert"
print("OK")

## 3) (İsteğe bağlı) Swap ekle — düşük RAM'de OOM'u azaltır

In [ ]:
#@title 16 GB swap oluştur (ücretsiz katmanda önerilir)
!fallocate -l 16G /content/swapfile 2>/dev/null || dd if=/dev/zero of=/content/swapfile bs=1M count=16384
!chmod 600 /content/swapfile && mkswap /content/swapfile && swapon /content/swapfile
!free -h

## 4) Google Drive'ı bağla + QNN SDK 2.28'i aç

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, glob
assert os.path.exists(QNN_SDK_ZIP_ON_DRIVE), (
    f"QNN SDK zip bulunamadı: {QNN_SDK_ZIP_ON_DRIVE}\n"
    "Qualcomm AI Engine Direct SDK 2.28'i Drive'a yükleyip yolunu 1. hücrede düzeltin.")

!mkdir -p /content/qairt && unzip -q -o "$QNN_SDK_ZIP_ON_DRIVE" -d /content/qairt
# SDK kök klasörünü otomatik bul (içinde bin/x86_64-linux-clang olan)
root = None
for d, _, _ in os.walk("/content/qairt"):
    if os.path.exists(os.path.join(d, "bin", "x86_64-linux-clang", "qnn-onnx-converter")):
        root = d; break
assert root, "QNN_SDK_ROOT bulunamadı (bin/x86_64-linux-clang/qnn-onnx-converter yok)."
os.environ["QNN_SDK_ROOT"] = root
print("QNN_SDK_ROOT =", root)
# QNN converter'ın python bağımlılıkları
!bash -c 'source "$QNN_SDK_ROOT/bin/envsetup.sh" 2>/dev/null; "$QNN_SDK_ROOT/bin/check-python-dependency" 2>/dev/null || true'

## 5) Modeli indir (civitai / HF / düz link)

In [ ]:
from google.colab import userdata
import os
for k in ("CIVITAI_TOKEN", "HF_TOKEN"):
    try:
        v = userdata.get(k)
        if v: os.environ[k] = v
    except Exception:
        pass

!python scripts/fetch_model.py --url "$SAFETENSORS_URL" --output work/input.safetensors

## 6) Dönüştür (uçtan uca)

**Uyarı:** Bu adım en uzunudur — çözünürlük ve tier başına **saatler** sürebilir. Colab oturumu kopmasın diye sekmeyi açık tutun.

In [ ]:
!chmod +x convert_all.sh scripts/*.sh
!./convert_all.sh work/input.safetensors "$MODEL_NAME" "$TIER" "$RESOLUTIONS"
import glob
zips = glob.glob("dist/*.zip")
print("Üretilen:", zips)

## 7) Hugging Face reposuna yükle (HF_REPO doluysa)

In [ ]:
import glob, os
zips = sorted(glob.glob("dist/*.zip"))
assert zips, "dist/ içinde zip yok — 6. adım başarısız olmuş olabilir."
zip_path = zips[-1]
if HF_REPO.strip():
    priv = "--private" if HF_PRIVATE else ""
    os.system(f'python scripts/upload_hf.py --repo "{HF_REPO}" --file "{zip_path}" {priv}')
else:
    print("HF_REPO boş — yükleme atlandı.")
print("ZIP:", zip_path)

## 8) (İsteğe bağlı) ZIP'i doğrudan bilgisayara indir

In [ ]:
from google.colab import files
import glob
zips = sorted(glob.glob("dist/*.zip"))
if zips:
    files.download(zips[-1])